# Funding signal: model experimentation

Signal-research phase for the funding-carry work. Loops a registry of candidate models (see `research/models.py`) through the same walk-forward CV and produces one comparison table (rank IC, R², directional, MSE). Persists per-run results under `research/results/` for cross-session comparison.

**One question, still:** does a model predict future funding better than dumb baselines? Ridge on the v1 feature set already showed a lift; this notebook widens the search across model class and hyperparameter, and (in later feature sets) adds regime indicators and kline aggregates.

Costs, sizing, and strategy tuning happen in the C++ engine against the shared-code-path sim. Nothing in this notebook is a PnL claim.

Canonical event frame: `[symbol, ts, realized_funding, premium]`, one row per (symbol, 8h ts). Label: `realized_cum` = sum of funding over the next N=24 intervals.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "research").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import research.features as F
import research.cv as CV
import research.models as M

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr
from datetime import datetime


DATA = REPO / "data" / "binance_historical"
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
           "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]

Funding CSVs: `data/binance_historical/<SYMBOL>/futures/funding/<SYMBOL>-fundingRate-YYYY-MM.csv`, no header, columns `symbol,tag,ts_ms,interval_hours,funding_rate`.

Premium-index klines: `data/binance_historical/<SYMBOL>/futures/premiumindex/`. Aggregate the interval's premium average up to each funding ts.

In [2]:
FUNDING_COLS = ["symbol", "tag", "ts_ms", "interval_h", "funding_rate"]

def load_funding(symbol: str) -> pd.DataFrame:
    d = DATA / symbol / "futures" / "funding"
    files = sorted(d.glob(f"{symbol}-fundingRate-*.csv"))
    if not files:
        raise FileNotFoundError(f"no funding csvs under {d}")
    df = pd.concat([pd.read_csv(f, header=None, names=FUNDING_COLS) for f in files], ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    df["symbol"] = symbol
    df = df[["symbol", "ts", "funding_rate"]].rename(columns={"funding_rate": "realized_funding"})
    return df.sort_values("ts").drop_duplicates("ts").reset_index(drop=True)


PREMIUM_COLS = ["ts_ms", "open", "high", "low", "close", "volume", "close_ms",
                "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]

def load_premium(symbol: str) -> pd.DataFrame:
    d = DATA / symbol / "futures" / "premiumindex"
    files = sorted(d.glob("*.csv"))
    if not files:
        return pd.DataFrame(columns=["ts_ms", "close"])
    parts = []
    for f in files:
        df = pd.read_csv(f, header=None, names=PREMIUM_COLS)
        parts.append(df[["ts_ms", "close"]])
    df = pd.concat(parts, ignore_index=True).sort_values("ts_ms").drop_duplicates("ts_ms").reset_index(drop=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    return df


def average_premium_up_to(premium: pd.DataFrame, funding_ts: pd.Series, interval_hours: int = 8) -> pd.Series:
    if premium.empty:
        return pd.Series(np.nan, index=range(len(funding_ts)))
    p = premium.sort_values("ts").reset_index(drop=True)
    win = pd.Timedelta(hours=interval_hours)
    out = np.full(len(funding_ts), np.nan)
    ts_vals = pd.to_datetime(funding_ts, utc=True).to_numpy()
    p_ts = p["ts"].to_numpy()
    p_close = p["close"].to_numpy(dtype=float)
    for i, end in enumerate(ts_vals):
        beg = end - win
        lo = np.searchsorted(p_ts, beg, side="right")
        hi = np.searchsorted(p_ts, end, side="right")
        if hi > lo:
            out[i] = p_close[lo:hi].mean()
    return pd.Series(out)


def load_events(symbols: list[str]) -> pd.DataFrame:
    parts = []
    for sym in symbols:
        f = load_funding(sym)
        p = load_premium(sym)
        f["premium"] = average_premium_up_to(p, f["ts"]).to_numpy()
        parts.append(f)
    return pd.concat(parts, ignore_index=True)


events = load_events(SYMBOLS)
print("rows:", len(events))
print(events.groupby("symbol")["ts"].agg(["min", "max", "count"]))
print("funding NaN:", events["realized_funding"].isna().sum(), " premium NaN:", events["premium"].isna().sum())

rows: 32955


                                      min                       max  count
symbol                                                                    
ADAUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
AVAXUSDT 2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
BNBUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
BTCUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
DOGEUSDT 2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
ETHUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
LINKUSDT 2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
LTCUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
SOLUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3363
XRPUSDT  2022-01-01 00:00:00.006000+00:00 2024-12-31 16:00:00+00:00   3288
funding NaN: 0  premium NaN: 2


Chained `F.add_*` calls. Drop rows with any NaN feature (warmup). Every feature is causal by construction.

In [3]:
HORIZON = 24  # predict next N=24 8h intervals, ~8 days ahead

feat = events.copy()
feat = F.add_funding_lag(feat, 1)
feat = F.add_funding_lag(feat, 2)
feat = F.add_funding_lag(feat, 3)
feat = F.add_funding_ewma(feat, halflife=2)
feat = F.add_funding_ewma(feat, halflife=6)
feat = F.add_funding_vol(feat, window=8)
feat = F.add_clamp_distance(feat)
feat = F.add_premium_trend(feat, span=3)
feat = F.add_cross_symbol_spread(feat, reference="BTCUSDT")
feat = F.add_basket_spread(feat)
feat = F.add_time_features(feat)
feat = F.add_cum_target(feat, horizon=HORIZON)

FEATURE_COLS = [c for c in feat.columns
                if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
print("features:", FEATURE_COLS)
print("target: realized_cum (sum of next", HORIZON, "8h funding prints)")
print("shape before dropna:", feat.shape)
feat_clean = feat.dropna(subset=FEATURE_COLS + ["realized_funding", "realized_cum"]).reset_index(drop=True)
print("shape after dropna:", feat_clean.shape)

features: ['funding_lag1', 'funding_lag2', 'funding_lag3', 'funding_ewma_h2', 'funding_ewma_h6', 'funding_vol_w8', 'clamp_distance', 'premium_trend_s3', 'cross_spread_vs_btcusdt', 'basket_spread', 'hour_of_day', 'day_of_week']
target: realized_cum (sum of next 24 8h funding prints)
shape before dropna: (32955, 17)
shape after dropna: (32608, 17)


## Model registry sweep

Loops `M.linear_registry_v1(HORIZON)` through the walk-forward folds. Registry:

- **persistence** — `24 * funding_lag1`, the benchmark.
- **ridge_a{0.01, 0.1, 1, 10, 100}** — L2 alpha sweep.
- **lasso_a{1e-6 ... 1e-2}** — L1 sweep. Alpha scale is much smaller than ridge's because the target scale is small; alphas above ~1e-2 zero out everything.
- **elasticnet_l1r{0.3, 0.5, 0.7}** — L1+L2 mix at a fixed alpha.
- **bayesian_ridge** — adaptive alpha, no manual tuning.

Metrics per fold: rank IC (Spearman), R², directional accuracy, MSE. R² is the decisive one, persistence is close to ridge on rank IC but blows up on R² because it over-extrapolates scale.

In [4]:
TARGET_COL = "realized_cum"

def eval_pred(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    ic, _ = spearmanr(y_pred, y_true)
    mse = float(np.mean((y_true - y_pred) ** 2))
    dir_acc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"r2": float(r2), "rank_ic": float(ic), "mse": mse, "dir_acc": dir_acc}

def run_registry(feat_clean, registry, feature_cols, target_col, n_folds=5):
    rows = []
    for fold_i, (tr, te) in enumerate(CV.walk_forward_splits(feat_clean, n_folds=n_folds, horizon=HORIZON, embargo=5)):
        train = feat_clean.iloc[tr]
        test = feat_clean.iloc[te]
        y_true = test[target_col].to_numpy()
        for name, fn in registry.items():
            y_pred = fn(train, test, feature_cols, target_col)
            m = eval_pred(y_true, y_pred)
            m["fold"] = fold_i
            m["model"] = name
            rows.append(m)
    return pd.DataFrame(rows)

registry = M.linear_registry_v1(horizon=HORIZON)
print(f"{len(registry)} models: {list(registry.keys())}")
metrics = run_registry(feat_clean, registry, FEATURE_COLS, TARGET_COL)

# Persist so we can compare across sessions.
results_dir = REPO / "research" / "results"
results_dir.mkdir(exist_ok=True)

15 models: ['persistence', 'ridge_a0.01', 'ridge_a0.1', 'ridge_a1.0', 'ridge_a10.0', 'ridge_a100.0', 'lasso_a1e-06', 'lasso_a1e-05', 'lasso_a0.0001', 'lasso_a0.001', 'lasso_a0.01', 'elasticnet_l1r0.3', 'elasticnet_l1r0.5', 'elasticnet_l1r0.7', 'bayesian_ridge']


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


## Summary by metric

Pivot the per-fold table into one view per metric. Judge signal quality on the rank-IC block first; R² and directional accuracy are supporting evidence.

In [5]:
mean_by_model = (
    metrics.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
           .mean()
           .sort_values("r2", ascending=False)
           .round(4)
)
print("=== mean across folds, ranked by R^2 ===")
print(mean_by_model)

print("\n=== r2 per fold (models ranked) ===")
print(metrics.pivot(index="fold", columns="model", values="r2")[mean_by_model.index].round(4))

print("\n=== rank_ic per fold (models ranked) ===")
print(metrics.pivot(index="fold", columns="model", values="rank_ic")[mean_by_model.index].round(4))

=== mean across folds, ranked by R^2 ===
                   rank_ic      r2  dir_acc  mse
model                                           
lasso_a1e-05        0.7071  0.4248   0.8720  0.0
ridge_a100.0        0.7061  0.4242   0.8718  0.0
bayesian_ridge      0.7060  0.4237   0.8714  0.0
elasticnet_l1r0.3   0.7081  0.4233   0.8749  0.0
lasso_a1e-06        0.7059  0.4231   0.8714  0.0
ridge_a10.0         0.7058  0.4229   0.8712  0.0
ridge_a1.0          0.7057  0.4227   0.8712  0.0
ridge_a0.1          0.7057  0.4227   0.8712  0.0
ridge_a0.01         0.7057  0.4226   0.8712  0.0
elasticnet_l1r0.5   0.7079  0.4170   0.8781  0.0
elasticnet_l1r0.7   0.7083  0.4143   0.8805  0.0
lasso_a0.0001       0.7083  0.4116   0.8849  0.0
lasso_a0.001        0.7411  0.2733   0.9061  0.0
lasso_a0.01            NaN -0.2241   0.8717  0.0
persistence         0.6726 -0.3109   0.8426  0.0

=== r2 per fold (models ranked) ===
model  lasso_a1e-05  ridge_a100.0  bayesian_ridge  elasticnet_l1r0.3  \
fold             

## Findings, v1 linear sweep

**Model class basically doesn't matter at v1.** Ridge, Lasso (low alpha), ElasticNet, and Bayesian Ridge all cluster within ~0.003 R² of each other at the top.

Leaderboard (mean across 5 folds, ranked by R²):

Ridge is nearly alpha insensitive. Standardised features + L2 with 12 features + 9 symbol dummies is genuinely stable.

Very light L1 or elasticnet edges out slightly (0.003 R² over ridge). Not a meaningful lift, but it's cheaper#, so worth carrying forward as the baseline.

`lasso_a0.001` is the interesting outlier. Heavy L1 shrinkage: R² drops to 0.29 but rank IC jumps to **0.741**. The model is predicting smaller magnitudes (worse variance-explained) but ordering better. Likely keeps only 2-3 features (funding_lag1, funding_ewma) with shrunk coefficients. For a strategy that gates on rank / direction, this could actually be better than the R² winner.

`lasso_a0.01` collapses. Predicts a constant, so rank IC is NaN. Alpha too high; expected.

v1 features are saturated for linear models. No model class tweak or regularisation strength is going to move R² past ~0.43 with this feature set. The next real lift comes from new features, not new models.

Two candidates carry forward:
- **elasticnet_l1r0.3** (R² winner, marginal over ridge but slightly sparser).
- **lasso_a0.001** (rank IC winner, worth watching if downstream strategy cares more about ranking than magnitude).

## v2 features, regime indicators

Add three regime signals to v1 and re-run the same linear registry.
- `funding_mean_w90` — rolling mean of funding over past 90 intervals (~30 days). Slow regime tracker.
- `funding_sign_w90` — sign of the same window. -1 / 0 / +1 categorical-style regime label.
- `funding_vol_rank_v30_r180` — rolling percentile rank of funding volatility (30-interval std, ranked over 180 intervals). Captures "high-vol vs low-vol regime" without hard thresholds.

All causal: computed on funding shifted by 1.

In [6]:
feat_v2 = events.copy()
feat_v2 = F.add_funding_lag(feat_v2, 1)
feat_v2 = F.add_funding_lag(feat_v2, 2)
feat_v2 = F.add_funding_lag(feat_v2, 3)
feat_v2 = F.add_funding_ewma(feat_v2, halflife=2)
feat_v2 = F.add_funding_ewma(feat_v2, halflife=6)
feat_v2 = F.add_funding_vol(feat_v2, window=8)
feat_v2 = F.add_clamp_distance(feat_v2)
feat_v2 = F.add_premium_trend(feat_v2, span=3)
feat_v2 = F.add_cross_symbol_spread(feat_v2, reference="BTCUSDT")
feat_v2 = F.add_basket_spread(feat_v2)
feat_v2 = F.add_time_features(feat_v2)

# v2 additions:
feat_v2 = F.add_funding_mean_window(feat_v2, window=90)
feat_v2 = F.add_funding_sign_window(feat_v2, window=90)
feat_v2 = F.add_funding_vol_rank(feat_v2, vol_window=30, rank_window=180)

feat_v2 = F.add_cum_target(feat_v2, horizon=HORIZON)

FEATURE_COLS_V2 = [c for c in feat_v2.columns
                   if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
print("v2 features:", FEATURE_COLS_V2)

feat_v2_clean = feat_v2.dropna(subset=FEATURE_COLS_V2 + ["realized_funding", "realized_cum"]).reset_index(drop=True)
print("shape after dropna:", feat_v2_clean.shape)

metrics_v2 = run_registry(feat_v2_clean, registry, FEATURE_COLS_V2, TARGET_COL)

mean_v2 = (
    metrics_v2.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
              .mean()
              .sort_values("r2", ascending=False)
              .round(4)
)
print("\n=== v2 mean across folds, ranked by R^2 ===")
print(mean_v2)

v2 features: ['funding_lag1', 'funding_lag2', 'funding_lag3', 'funding_ewma_h2', 'funding_ewma_h6', 'funding_vol_w8', 'clamp_distance', 'premium_trend_s3', 'cross_spread_vs_btcusdt', 'basket_spread', 'hour_of_day', 'day_of_week', 'funding_mean_w90', 'funding_sign_w90', 'funding_vol_rank_v30_r180']
shape after dropna: (32058, 20)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)



=== v2 mean across folds, ranked by R^2 ===
                   rank_ic      r2  dir_acc  mse
model                                           
lasso_a1e-05        0.6830  0.4160   0.8783  0.0
ridge_a100.0        0.6839  0.4145   0.8770  0.0
bayesian_ridge      0.6835  0.4139   0.8766  0.0
elasticnet_l1r0.3   0.6810  0.4135   0.8828  0.0
lasso_a1e-06        0.6825  0.4130   0.8762  0.0
lasso_a0.0001       0.6843  0.4127   0.8951  0.0
ridge_a10.0         0.6823  0.4125   0.8759  0.0
ridge_a1.0          0.6819  0.4121   0.8758  0.0
ridge_a0.1          0.6819  0.4121   0.8758  0.0
ridge_a0.01         0.6819  0.4121   0.8758  0.0
elasticnet_l1r0.5   0.6798  0.4102   0.8860  0.0
elasticnet_l1r0.7   0.6801  0.4099   0.8901  0.0
lasso_a0.001        0.7292  0.2646   0.9091  0.0
lasso_a0.01            NaN -0.2256   0.8755  0.0
persistence         0.6643 -0.2724   0.8433  0.0


Regime features added negligible R² and slightly hurt rank IC (0.729 vs v1 0.741 for lasso_a0.001; other models similar). The regime information was already implicit in the existing lags/EWMAs.

Regime as a linear feature is a dead end. If regime matters, it matters through *interactions* (regime × lag) or a *gate* (HMM on top). Both are later steps.
The linear ceiling on funding-only features is ~0.43 R².

## v3 features, kline aggregates

Per-8h aggregates of the 1-minute futures klines over the interval ending at each funding ts.

- `taker_imbalance` — `(2·taker_buy_base − volume) / volume`. Direct proxy for aggressive leverage / positioning.
- `realized_return` — 8h return on close.
- `realized_vol_1m` — std of 1-min log-returns within the 8h.
- `high_low_range` — `(high − low) / close_open`. Simple range proxy for intraday dispersion.

Kline data is 10 symbols × ~1095 daily 1-min CSVs. Loading + aggregation is cached to `research/results/kline_aggregates_8h.parquet` after the first run.

In [7]:
KLINE_COLS = ["symbol", "tag", "ts_ms", "open", "high", "low", "close", "volume",
              "close_ms", "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def load_symbol_klines(symbol: str) -> pd.DataFrame:
    d = DATA / symbol / "futures" / "klines"
    files = sorted(d.glob(f"{symbol}-1m-*.csv"))
    if not files:
        raise FileNotFoundError(f"no kline csvs under {d}")
    parts = []
    for f in files:
        df = pd.read_csv(f, header=None, names=KLINE_COLS,
                         usecols=["ts_ms", "high", "low", "close", "volume", "taker_buy_base"])
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


cache_path = results_dir / "kline_aggregates_8h.pkl"
if cache_path.exists():
    print(f"loading cached aggregates from {cache_path.relative_to(REPO)}")
    kline_agg = pd.read_pickle(cache_path)
else:
    print("aggregating klines per symbol (one-time)...")
    parts = []
    for sym in SYMBOLS:
        print(f"  {sym}...", end=" ", flush=True)
        klines = load_symbol_klines(sym)
        ts_sym = feat_v2[feat_v2["symbol"] == sym]["ts"]
        agg = F.aggregate_klines_to_intervals(klines, ts_sym)
        agg["symbol"] = sym
        parts.append(agg)
        del klines
        print(f"{len(agg)} rows")
    kline_agg = pd.concat(parts, ignore_index=True)
    kline_agg.to_pickle(cache_path)
    print(f"cached -> {cache_path.relative_to(REPO)}")

print("kline aggregate columns:", list(kline_agg.columns))
print("NaN per column:")
print(kline_agg.isna().sum())

loading cached aggregates from research/results/kline_aggregates_8h.pkl
kline aggregate columns: ['ts', 'taker_imbalance', 'realized_return', 'realized_vol_1m', 'high_low_range', 'symbol']
NaN per column:
ts                  0
taker_imbalance    10
realized_return    10
realized_vol_1m    10
high_low_range     10
symbol              0
dtype: int64


In [8]:
feat_v3 = feat_v2.merge(kline_agg, on=["symbol", "ts"], how="left")

FEATURE_COLS_V3 = [c for c in feat_v3.columns
                   if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
print("v3 features:", FEATURE_COLS_V3)

feat_v3_clean = feat_v3.dropna(subset=FEATURE_COLS_V3 + ["realized_funding", "realized_cum"]).reset_index(drop=True)
print("shape after dropna:", feat_v3_clean.shape)

metrics_v3 = run_registry(feat_v3_clean, registry, FEATURE_COLS_V3, TARGET_COL)

mean_v3 = (
    metrics_v3.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
              .mean()
              .sort_values("r2", ascending=False)
              .round(4)
)
print("\n=== v3 mean across folds, ranked by R^2 ===")
print(mean_v3)

v3 features: ['funding_lag1', 'funding_lag2', 'funding_lag3', 'funding_ewma_h2', 'funding_ewma_h6', 'funding_vol_w8', 'clamp_distance', 'premium_trend_s3', 'cross_spread_vs_btcusdt', 'basket_spread', 'hour_of_day', 'day_of_week', 'funding_mean_w90', 'funding_sign_w90', 'funding_vol_rank_v30_r180', 'taker_imbalance', 'realized_return', 'realized_vol_1m', 'high_low_range']
shape after dropna: (32058, 24)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)



=== v3 mean across folds, ranked by R^2 ===
                   rank_ic      r2  dir_acc  mse
model                                           
ridge_a100.0        0.6749  0.4071   0.8761  0.0
lasso_a0.0001       0.6774  0.4070   0.8960  0.0
bayesian_ridge      0.6747  0.4067   0.8761  0.0
lasso_a1e-05        0.6740  0.4066   0.8784  0.0
elasticnet_l1r0.3   0.6718  0.4059   0.8828  0.0
lasso_a1e-06        0.6736  0.4054   0.8758  0.0
ridge_a10.0         0.6734  0.4052   0.8758  0.0
ridge_a1.0          0.6731  0.4048   0.8756  0.0
ridge_a0.1          0.6731  0.4048   0.8755  0.0
ridge_a0.01         0.6731  0.4048   0.8755  0.0
elasticnet_l1r0.5   0.6703  0.4033   0.8871  0.0
elasticnet_l1r0.7   0.6716  0.4022   0.8910  0.0
lasso_a0.001        0.7292  0.2646   0.9091  0.0
lasso_a0.01            NaN -0.2256   0.8755  0.0
persistence         0.6643 -0.2724   0.8433  0.0


Kline aggregates marginally *hurt* linear model. `lasso_a0.001`'s rank IC is bitwise identical to v2, L1 zeroed out the new features entirely.

Not that the features are wrong, but as *standalone linear predictors of next-24-interval funding* they add noise more than signal. Taker imbalance and realized vol are correlated with the funding vol / EWMA terms ridge already has, and their marginal information is 8h-aggregated and noisy.

## v4 features, interactions

Handpicked cross-terms, not a full polynomial expansion (which would generate 171 pairs from 19 features and mostly add noise).

- `taker_imb_x_ewma2` = `taker_imbalance × funding_ewma_h2` — positioning direction aligned with recent trend.
- `taker_imb_x_lag1`  = `taker_imbalance × funding_lag1` — positioning × most recent print.
- `vol1m_x_clamp`     = `realized_vol_1m × clamp_distance` — price vol conditional on premium level.
- `range_x_ewma2`     = `high_low_range × funding_ewma_h2` — intraday dispersion × trend.
- `regime_x_lag1`     = `funding_sign_w90 × funding_lag1` — regime toggle applied to short-horizon signal.
- `basket_x_lag1`     = `basket_spread × funding_lag1` — relative position within basket × absolute signal.
- `lag1_sq`           = `funding_lag1²` — nonlinear response to recent funding.

In [9]:
feat_v4 = feat_v3.copy()
feat_v4["taker_imb_x_ewma2"] = feat_v4["taker_imbalance"] * feat_v4["funding_ewma_h2"]
feat_v4["taker_imb_x_lag1"]  = feat_v4["taker_imbalance"] * feat_v4["funding_lag1"]
feat_v4["vol1m_x_clamp"]     = feat_v4["realized_vol_1m"] * feat_v4["clamp_distance"]
feat_v4["range_x_ewma2"]     = feat_v4["high_low_range"]  * feat_v4["funding_ewma_h2"]
feat_v4["regime_x_lag1"]     = feat_v4["funding_sign_w90"] * feat_v4["funding_lag1"]
feat_v4["basket_x_lag1"]     = feat_v4["basket_spread"]   * feat_v4["funding_lag1"]
feat_v4["lag1_sq"]           = feat_v4["funding_lag1"] ** 2

FEATURE_COLS_V4 = [c for c in feat_v4.columns
                   if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
print("v4 features:", FEATURE_COLS_V4)

feat_v4_clean = feat_v4.dropna(subset=FEATURE_COLS_V4 + ["realized_funding", "realized_cum"]).reset_index(drop=True)
print("shape after dropna:", feat_v4_clean.shape)

metrics_v4 = run_registry(feat_v4_clean, registry, FEATURE_COLS_V4, TARGET_COL)

mean_v4 = (
    metrics_v4.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
              .mean()
              .sort_values("r2", ascending=False)
              .round(4)
)
print("\n=== v4 mean across folds, ranked by R^2 ===")
print(mean_v4)

v4 features: ['funding_lag1', 'funding_lag2', 'funding_lag3', 'funding_ewma_h2', 'funding_ewma_h6', 'funding_vol_w8', 'clamp_distance', 'premium_trend_s3', 'cross_spread_vs_btcusdt', 'basket_spread', 'hour_of_day', 'day_of_week', 'funding_mean_w90', 'funding_sign_w90', 'funding_vol_rank_v30_r180', 'taker_imbalance', 'realized_return', 'realized_vol_1m', 'high_low_range', 'taker_imb_x_ewma2', 'taker_imb_x_lag1', 'vol1m_x_clamp', 'range_x_ewma2', 'regime_x_lag1', 'basket_x_lag1', 'lag1_sq']
shape after dropna: (32058, 31)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)



=== v4 mean across folds, ranked by R^2 ===
                   rank_ic      r2  dir_acc  mse
model                                           
elasticnet_l1r0.7   0.7167  0.4540   0.8832  0.0
elasticnet_l1r0.5   0.7185  0.4527   0.8801  0.0
lasso_a0.0001       0.7075  0.4445   0.8906  0.0
elasticnet_l1r0.3   0.7169  0.4445   0.8755  0.0
lasso_a1e-05        0.7130  0.4250   0.8698  0.0
ridge_a100.0        0.7109  0.4196   0.8683  0.0
bayesian_ridge      0.7100  0.4086   0.8663  0.0
lasso_a1e-06        0.7100  0.4081   0.8663  0.0
ridge_a10.0         0.7098  0.4069   0.8664  0.0
ridge_a1.0          0.7092  0.4031   0.8656  0.0
ridge_a0.1          0.7091  0.4024   0.8656  0.0
ridge_a0.01         0.7090  0.4023   0.8656  0.0
lasso_a0.001        0.7292  0.2646   0.9091  0.0
lasso_a0.01            NaN -0.2256   0.8755  0.0
persistence         0.6643 -0.2724   0.8433  0.0


## GBM on v3 features

Now that interactions clearly help in the linear model, test whether a tree model finds those interactions on its own from *raw* v3 features. If GBM on v3 matches or beats elasticnet on v4, the tree is discovering the same structure; if it doesn't, the handpicked interactions carry information a default GBM doesn't find.

Three GBM configs: default, tight (fewer leaves + more min_data + slower learning rate), tighter (same direction more so).

In [10]:
gbm_reg = M.gbm_registry()
print(f"gbm configs: {list(gbm_reg.keys())}")

metrics_gbm_v3 = run_registry(feat_v3_clean, gbm_reg, FEATURE_COLS_V3, TARGET_COL)
mean_gbm_v3 = (
    metrics_gbm_v3.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
                  .mean().sort_values("r2", ascending=False).round(4)
)
print("=== GBM on v3, mean across folds ===")
print(mean_gbm_v3)

# Also run GBM on v4 for the direct comparison — does GBM further benefit from
# handpicked interactions, or is it happy with raw features?
metrics_gbm_v4 = run_registry(feat_v4_clean, gbm_reg, FEATURE_COLS_V4, TARGET_COL)
mean_gbm_v4 = (
    metrics_gbm_v4.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
                  .mean().sort_values("r2", ascending=False).round(4)
)
print("\n=== GBM on v4, mean across folds ===")
print(mean_gbm_v4)

gbm configs: ['gbm_default', 'gbm_tight', 'gbm_tighter']


=== GBM on v3, mean across folds ===
             rank_ic      r2  dir_acc  mse
model                                     
gbm_tighter   0.7038  0.3146   0.8899  0.0
gbm_tight     0.6986  0.2311   0.8867  0.0
gbm_default   0.6837 -0.0553   0.8859  0.0



=== GBM on v4, mean across folds ===
             rank_ic      r2  dir_acc  mse
model                                     
gbm_tighter   0.7035  0.3157   0.8888  0.0
gbm_tight     0.6990  0.2399   0.8874  0.0
gbm_default   0.6823 -0.0309   0.8873  0.0


### GBM read

GBM underperforms linear at every regularisation level, on both v3 (raw) and v4 (with handpicked interactions).

- `gbm_tighter` tops out at R² 0.32 (v3) / 0.32 (v4). Linear elasticnet on v4 gets 0.45.
- `gbm_default` overfits — R² near zero out-of-fold on both feature sets. Default LightGBM is far too flexible for ~30k autocorrelated rows on this target.
- GBM barely benefits from handpicked interactions being added (v3→v4 lift of 0.001 R²). Consistent with the intuition that tree splits *are* pair interactions — but the ones GBM finds greedily from noise are worse than the ones we handed to L1 from theory.
- Directional accuracy is marginally higher for GBM (~0.89 vs ~0.87 linear), but that's the least informative metric here.

**Winner on v4: elasticnet_l1r0.7, R² 0.4540, rank IC 0.7167.**

The signal appears to be linear + a small set of known interactions.

**Update (2026-09-07):** the three-config sweep above is under-regularised and uses no early stopping. Re-running with proper per-fold early stopping and a five-config grid on v5 features (`gbm_es_v5` cell below) flips the read on rank IC and dir_acc: best config lifts rank IC 0.72 → 0.76 and dir_acc 0.88 → 0.92, gives up 0.03 R² to the elasticnet. Treat this section as an exhaustion-of-defaults check, not the GBM verdict.


## v5 features, cross-symbol z-scores and ranks

How this symbol's funding compares to the cross-sectional basket at the same ts. `basket_spread` already exists (raw difference); v5 adds scale-normalised versions.

- `basket_z_funding_lag1` — (funding_lag1 − cross_mean) / cross_std at each ts. Cross-sectional z-score.
- `basket_z_funding_ewma_h6` — same but on the slower funding EWMA.
- `basket_rank_funding_lag1` — cross-sectional percentile rank at each ts (0-1). Robust to outliers where a z-score would blow up.

Also add interactions with recent funding to see if they gain traction the way v4 pair-terms did:
- `basket_z_x_lag1` = `basket_z_funding_lag1 × funding_lag1`
- `basket_rank_x_lag1` = `basket_rank_funding_lag1 × funding_lag1`

In [11]:
feat_v5 = feat_v4.copy()
feat_v5 = F.add_basket_zscore(feat_v5, source="funding_lag1")
feat_v5 = F.add_basket_zscore(feat_v5, source="funding_ewma_h6")
feat_v5 = F.add_basket_rank(feat_v5, source="funding_lag1")

feat_v5["basket_z_x_lag1"]    = feat_v5["basket_z_funding_lag1"]    * feat_v5["funding_lag1"]
feat_v5["basket_rank_x_lag1"] = feat_v5["basket_rank_funding_lag1"] * feat_v5["funding_lag1"]

FEATURE_COLS_V5 = [c for c in feat_v5.columns
                   if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
print("v5 features:", FEATURE_COLS_V5)

feat_v5_clean = feat_v5.dropna(subset=FEATURE_COLS_V5 + ["realized_funding", "realized_cum"]).reset_index(drop=True)
print("shape after dropna:", feat_v5_clean.shape)

metrics_v5 = run_registry(feat_v5_clean, registry, FEATURE_COLS_V5, TARGET_COL)

mean_v5 = (
    metrics_v5.groupby("model")[["rank_ic", "r2", "dir_acc", "mse"]]
              .mean()
              .sort_values("r2", ascending=False)
              .round(4)
)
print("\n=== v5 mean across folds, ranked by R^2 ===")
print(mean_v5)

v5 features: ['funding_lag1', 'funding_lag2', 'funding_lag3', 'funding_ewma_h2', 'funding_ewma_h6', 'funding_vol_w8', 'clamp_distance', 'premium_trend_s3', 'cross_spread_vs_btcusdt', 'basket_spread', 'hour_of_day', 'day_of_week', 'funding_mean_w90', 'funding_sign_w90', 'funding_vol_rank_v30_r180', 'taker_imbalance', 'realized_return', 'realized_vol_1m', 'high_low_range', 'taker_imb_x_ewma2', 'taker_imb_x_lag1', 'vol1m_x_clamp', 'range_x_ewma2', 'regime_x_lag1', 'basket_x_lag1', 'lag1_sq', 'basket_z_funding_lag1', 'basket_z_funding_ewma_h6', 'basket_rank_funding_lag1', 'basket_z_x_lag1', 'basket_rank_x_lag1']
shape after dropna: (32058, 36)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)


/tmp/ipykernel_661717/3472656929.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(y_pred, y_true)



=== v5 mean across folds, ranked by R^2 ===
                   rank_ic      r2  dir_acc  mse
model                                           
elasticnet_l1r0.7   0.7199  0.4545   0.8831  0.0
elasticnet_l1r0.5   0.7226  0.4534   0.8801  0.0
elasticnet_l1r0.3   0.7214  0.4456   0.8751  0.0
lasso_a0.0001       0.7095  0.4435   0.8893  0.0
lasso_a1e-05        0.7166  0.4254   0.8711  0.0
ridge_a100.0        0.7136  0.4176   0.8705  0.0
bayesian_ridge      0.7121  0.4060   0.8684  0.0
lasso_a1e-06        0.7119  0.4044   0.8682  0.0
ridge_a10.0         0.7117  0.4035   0.8679  0.0
ridge_a1.0          0.7112  0.3991   0.8677  0.0
ridge_a0.1          0.7110  0.3980   0.8677  0.0
ridge_a0.01         0.7110  0.3978   0.8677  0.0
lasso_a0.001        0.7292  0.2646   0.9091  0.0
lasso_a0.01            NaN -0.2256   0.8755  0.0
persistence         0.6643 -0.2724   0.8433  0.0


In [12]:
import lightgbm as lgb
from research.models import _with_symbol_dummies

def _fit_gbm_es(train, feature_cols, target_col, params, val_frac=0.2):
    X_all, cols = _with_symbol_dummies(train, feature_cols)
    y_all = train[target_col].to_numpy()
    n = len(y_all)
    split = int(n * (1 - val_frac))
    p = dict(objective="regression", verbose=-1, feature_fraction=0.9,
             bagging_fraction=0.9, bagging_freq=5)
    p.update(params)
    dtr = lgb.Dataset(X_all[:split], label=y_all[:split], feature_name=cols)
    dval = lgb.Dataset(X_all[split:], label=y_all[split:], reference=dtr)
    booster = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dval],
                        callbacks=[lgb.early_stopping(100, verbose=False),
                                   lgb.log_evaluation(0)])
    return booster, cols

def run_gbm_es(feat_clean, feature_cols, target_col, grid, n_folds=5):
    rows = []
    for fold_i, (tr, te) in enumerate(
            CV.walk_forward_splits(feat_clean, n_folds=n_folds, horizon=HORIZON, embargo=5)):
        train = feat_clean.iloc[tr]
        test = feat_clean.iloc[te]
        y_true = test[target_col].to_numpy()
        for name, params in grid.items():
            booster, _ = _fit_gbm_es(train, feature_cols, target_col, params)
            Xte, _ = _with_symbol_dummies(test, feature_cols)
            y_pred = booster.predict(Xte, num_iteration=booster.best_iteration)
            m = eval_pred(y_true, y_pred)
            m["fold"] = fold_i
            m["model"] = name
            m["best_iter"] = int(booster.best_iteration)
            rows.append(m)
    return pd.DataFrame(rows)

gbm_grid = {
    "gbm_lr0.05_l31": {"learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200},
    "gbm_lr0.02_l15": {"learning_rate": 0.02, "num_leaves": 15, "min_data_in_leaf": 500},
    "gbm_lr0.02_l7":  {"learning_rate": 0.02, "num_leaves": 7,  "min_data_in_leaf": 1000},
    "gbm_lr0.01_l31": {"learning_rate": 0.01, "num_leaves": 31, "min_data_in_leaf": 200},
    "gbm_lr0.01_l63": {"learning_rate": 0.01, "num_leaves": 63, "min_data_in_leaf": 100},
}
metrics_gbm_v5 = run_gbm_es(feat_v5_clean, FEATURE_COLS_V5, TARGET_COL, gbm_grid)
mean_gbm_v5 = (
    metrics_gbm_v5.groupby("model")[["rank_ic", "r2", "dir_acc", "mse", "best_iter"]]
                  .mean().sort_values("r2", ascending=False).round(4)
)
print("=== GBM (early stopping) on v5, horizon-fixed CV ===")
print(mean_gbm_v5)

print("\n=== v5 ridge/elasticnet winners for comparison ===")
print(mean_v5.head(6))


=== GBM (early stopping) on v5, horizon-fixed CV ===
                rank_ic      r2  dir_acc  mse  best_iter
model                                                   
gbm_lr0.02_l7    0.7501  0.4346   0.9183  0.0      760.8
gbm_lr0.02_l15   0.7577  0.4275   0.9163  0.0      378.8
gbm_lr0.01_l31   0.7571  0.4122   0.9156  0.0      432.8
gbm_lr0.05_l31   0.7555  0.4109   0.9149  0.0       90.2
gbm_lr0.01_l63   0.7494  0.3694   0.9144  0.0      413.2

=== v5 ridge/elasticnet winners for comparison ===
                   rank_ic      r2  dir_acc  mse
model                                           
elasticnet_l1r0.7   0.7199  0.4545   0.8831  0.0
elasticnet_l1r0.5   0.7226  0.4534   0.8801  0.0
elasticnet_l1r0.3   0.7214  0.4456   0.8751  0.0
lasso_a0.0001       0.7095  0.4435   0.8893  0.0
lasso_a1e-05        0.7166  0.4254   0.8711  0.0
ridge_a100.0        0.7136  0.4176   0.8705  0.0


Marginal lift (+0.001 R², +0.003 rank IC) over v4. Cross-symbol z-scores add negligible information beyond `basket_spread` — a z-score is just a variance-normalised spread, and its added information (basket dispersion) doesn't move the linear predictor. Same winner, same shape.

### Feature ladder standing

| feature set                        | best model              | R²     | rank IC |
|------------------------------------|-------------------------|--------|---------|
| v1                                 | lasso_a1e-05            | 0.4248 | 0.7071  |
| v2 +regime                         | lasso_a1e-05            | 0.4160 | 0.6830  |
| v3 +klines                         | ridge_a100.0            | 0.4071 | 0.6749  |
| **v4 +interactions**               | elasticnet_l1r0.7       | 0.4540 | 0.7167  |
| **v5 +basket z/rank**              | **elasticnet_l1r0.7**   | **0.4545** | 0.7199 |
| GBM tighter (old sweep, no ES)     | gbm_tighter             | 0.32   | 0.70    |
| **GBM lr0.02 l15 (ES, v5)**        | **gbm_lr0.02_l15**      | 0.4275 | **0.7577** |

**Linear-family ceiling on this target sits at R² ≈ 0.45.** Feature ladder is exhausted for linear models. Properly early-stopped LightGBM on v5 gives up 0.03 R² but lifts rank IC by 0.04 (0.72 → 0.76) and dir_acc by 0.03 (0.88 → 0.92); if the strategy gates on rank/direction rather than magnitude, GBM is a live option.


Now that's a much cleaner read.

In-gate lift is real for gate_sign, marginal for mean_low_5e-5, harmful for mean_low_1e-4.

## Per-symbol elasticnet

Fit a private elasticnet per symbol per fold on v5 features (no symbol dummies, since the model is symbol-specific). Compare against the pooled v5 winner globally and per-symbol.

In [13]:
def fit_predict_elasticnet(train_df, test_df, feature_cols, target_col, alpha=1e-4, l1_ratio=0.7):
    Xtr = train_df[feature_cols].to_numpy(dtype=float)
    ytr = train_df[target_col].to_numpy()
    pipe = Pipeline([("sc", StandardScaler()),
                     ("m", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))])
    pipe.fit(Xtr, ytr)
    return pipe.predict(test_df[feature_cols].to_numpy(dtype=float))

def run_per_symbol(feat_clean, feature_cols, target_col, n_folds=5):
    """Fit one elasticnet per symbol per fold. Aggregates predictions and evaluates both pooled and per-symbol."""
    all_preds = []
    for fold_i, (tr, te) in enumerate(CV.walk_forward_splits(feat_clean, n_folds=n_folds, horizon=HORIZON, embargo=5)):
        train = feat_clean.iloc[tr]
        test = feat_clean.iloc[te]
        for sym, test_g in test.groupby("symbol", sort=False):
            train_g = train[train["symbol"] == sym]
            if len(train_g) < 200:  # too little history
                yhat = np.zeros(len(test_g))
            else:
                yhat = fit_predict_elasticnet(train_g, test_g, feature_cols, target_col)
            df = test_g[["symbol", "ts", target_col]].copy()
            df["yhat"] = yhat
            df["fold"] = fold_i
            all_preds.append(df)
    return pd.concat(all_preds, ignore_index=True)

preds_per_symbol = run_per_symbol(feat_v5_clean, FEATURE_COLS_V5, TARGET_COL)

# Aggregate metrics globally (pooled across all symbols and folds)
global_metrics = []
for fold_i in sorted(preds_per_symbol["fold"].unique()):
    fp = preds_per_symbol[preds_per_symbol["fold"] == fold_i]
    m = eval_pred(fp[TARGET_COL].to_numpy(), fp["yhat"].to_numpy())
    m["fold"] = fold_i
    global_metrics.append(m)
global_metrics = pd.DataFrame(global_metrics)

print("=== per-symbol elasticnet, mean across folds ===")
print(global_metrics[["rank_ic", "r2", "dir_acc"]].mean().round(4))

# Per-symbol breakdown, averaged across folds
per_sym_metrics = []
for (sym, fold_i), g in preds_per_symbol.groupby(["symbol", "fold"]):
    m = eval_pred(g[TARGET_COL].to_numpy(), g["yhat"].to_numpy())
    m["symbol"] = sym
    m["fold"] = fold_i
    per_sym_metrics.append(m)
per_sym_metrics = pd.DataFrame(per_sym_metrics)

per_sym_mean = per_sym_metrics.groupby("symbol")[["rank_ic", "r2", "dir_acc"]].mean().round(4)
print("\n=== per-symbol elasticnet, per-symbol mean R2 & rank IC ===")
print(per_sym_mean.sort_values("r2", ascending=False))

=== per-symbol elasticnet, mean across folds ===
rank_ic    0.7082
r2         0.2657
dir_acc    0.8907
dtype: float64

=== per-symbol elasticnet, per-symbol mean R2 & rank IC ===
          rank_ic      r2  dir_acc
symbol                            
BTCUSDT    0.7631  0.3965   0.9670
ADAUSDT    0.7251  0.3945   0.9114
XRPUSDT    0.6988  0.3759   0.9474
AVAXUSDT   0.7295  0.3258   0.8526
DOGEUSDT   0.6615  0.2604   0.9924
ETHUSDT    0.7002  0.2235   0.9835
LTCUSDT    0.7480  0.1926   0.9871
BNBUSDT    0.5331  0.1131   0.6984
LINKUSDT   0.7365 -0.0889   0.9755
SOLUSDT    0.4711 -1.5088   0.5920


## Regime gating (fair evaluation)

The gate is a *strategy* decision ("don't trade in idle regimes"), not a model decision. Errors on skipped rows are strategy PnL, not model quality — those belong in the C++ engine. So evaluate the model **on the trade subset only** (rows the gate says to act on), and compare against the ungated model *on the same subset*.

Alongside the in-gate metric, print a skipped-rows summary: how big was actual funding there, and what did the model predict? Together those tell us:
- **Gate aligned with model** — small predictions and small realised funding in skipped rows. Gate adds nothing to *model* quality, but a strategy can still use it to save fees on near-zero predictions.
- **Gate overrides model** — model predicted meaningful values in skipped rows, or funding was meaningful there. Gate is throwing away signal.

Gates tested:
- `gate_sign` — trade only where `funding_sign_w90 > 0`.
- `gate_mean_low_5e-5` — trade only where `funding_mean_w90 > 5e-5`.
- `gate_mean_low_1e-4` — trade only where `funding_mean_w90 > 1e-4`.


In [14]:
def fit_ungated_predictions(feat_clean, feature_cols, target_col, alpha=1e-4, l1_ratio=0.7, n_folds=5):
    """Fit v5 elasticnet per fold and return a frame with y_true, y_pred, fold, plus
    the gate columns needed downstream. No gating applied here."""
    from research.models import _with_symbol_dummies
    rows = []
    for fold_i, (tr, te) in enumerate(CV.walk_forward_splits(feat_clean, n_folds=n_folds, horizon=HORIZON, embargo=5)):
        train = feat_clean.iloc[tr]
        test  = feat_clean.iloc[te]
        Xtr, _ = _with_symbol_dummies(train, feature_cols)
        ytr = train[target_col].to_numpy()
        pipe = Pipeline([("sc", StandardScaler()),
                         ("m", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))])
        pipe.fit(Xtr, ytr)
        Xte, _ = _with_symbol_dummies(test, feature_cols)
        yhat = pipe.predict(Xte)
        out = test[["symbol", "ts", target_col, "funding_sign_w90", "funding_mean_w90"]].copy()
        out["y_pred"] = yhat
        out["fold"] = fold_i
        rows.append(out)
    return pd.concat(rows, ignore_index=True)


preds_v5 = fit_ungated_predictions(feat_v5_clean, FEATURE_COLS_V5, TARGET_COL)

gates = {
    "gate_none":            np.ones(len(preds_v5), dtype=bool),
    "gate_sign":            (preds_v5["funding_sign_w90"] > 0).to_numpy(),
    "gate_mean_low_5e-5":   (preds_v5["funding_mean_w90"] > 5e-5).to_numpy(),
    "gate_mean_low_1e-4":   (preds_v5["funding_mean_w90"] > 1e-4).to_numpy(),
}


def metrics_on_subset(sub: pd.DataFrame) -> dict:
    if len(sub) < 2:
        return {"r2": float("nan"), "rank_ic": float("nan"), "dir_acc": float("nan"), "n": len(sub)}
    y = sub[TARGET_COL].to_numpy()
    p = sub["y_pred"].to_numpy()
    return {**eval_pred(y, p), "n": len(sub)}


print("=== In-gate metrics (model evaluated only on rows the gate keeps) ===")
rows = []
for name, mask in gates.items():
    sub = preds_v5[mask]
    m = metrics_on_subset(sub)
    m["gate"] = name
    m["kept_frac"] = float(mask.mean())
    rows.append(m)
print(pd.DataFrame(rows)[["gate", "kept_frac", "n", "r2", "rank_ic", "dir_acc"]].round(4))

print("\n=== Skipped-rows summary ===")
print("Compare mean(y) and mean(pred) in the rows the gate throws away.")
print("If both are near zero, gate is aligned with the model (fee-saver at best).")
print("If either has appreciable magnitude, gate is overriding real signal.\n")
rows = []
for name, mask in gates.items():
    if name == "gate_none":
        continue
    skip = preds_v5[~mask]
    rows.append({
        "gate": name,
        "n_skipped": len(skip),
        "skip_frac": float((~mask).mean()),
        "y_mean_skipped": float(skip[TARGET_COL].mean()) if len(skip) else float("nan"),
        "y_abs_mean_skipped": float(skip[TARGET_COL].abs().mean()) if len(skip) else float("nan"),
        "pred_mean_skipped": float(skip["y_pred"].mean()) if len(skip) else float("nan"),
        "pred_abs_mean_skipped": float(skip["y_pred"].abs().mean()) if len(skip) else float("nan"),
    })
print(pd.DataFrame(rows).round(6))


=== In-gate metrics (model evaluated only on rows the gate keeps) ===
                 gate  kept_frac      n      r2  rank_ic  dir_acc
0           gate_none     1.0000  22448  0.5049   0.7854   0.8831
1           gate_sign     0.8724  19583  0.5127   0.7647   0.9074
2  gate_mean_low_5e-5     0.6705  15052  0.4835   0.7620   0.9344
3  gate_mean_low_1e-4     0.2832   6357  0.3991   0.7606   0.9659

=== Skipped-rows summary ===
Compare mean(y) and mean(pred) in the rows the gate throws away.
If both are near zero, gate is aligned with the model (fee-saver at best).
If either has appreciable magnitude, gate is overriding real signal.

                 gate  n_skipped  skip_frac  y_mean_skipped  \
0           gate_sign       2865   0.127628       -0.001718   
1  gate_mean_low_5e-5       7396   0.329473       -0.000047   
2  gate_mean_low_1e-4      16091   0.716812        0.000967   

   y_abs_mean_skipped  pred_mean_skipped  pred_abs_mean_skipped  
0            0.002918          -0.001507 

### Pooled vs per-symbol, side-by-side

Regroup the pooled v5 elasticnet's per-fold predictions (`preds_v5` from above) by symbol and put them next to the per-symbol model's own per-symbol numbers. Answers whether pooling with symbol dummies + basket features actually helps, or whether fitting a private model per symbol would do as well.


In [15]:
# Pooled v5 elasticnet (from fit_ungated_predictions above) broken down per
# symbol, joined against the per-symbol model's own per-symbol numbers.
pooled_per_sym = []
for sym, g in preds_v5.groupby("symbol"):
    m = eval_pred(g[TARGET_COL].to_numpy(), g["y_pred"].to_numpy())
    m["symbol"] = sym
    pooled_per_sym.append(m)
pooled_per_sym = pd.DataFrame(pooled_per_sym).set_index("symbol")[["rank_ic", "r2", "dir_acc"]]

cmp = pooled_per_sym.add_prefix("pooled_").join(per_sym_mean.add_prefix("per_sym_"))
cmp["d_rank_ic"] = (cmp["per_sym_rank_ic"] - cmp["pooled_rank_ic"]).round(4)
cmp["d_r2"]      = (cmp["per_sym_r2"]      - cmp["pooled_r2"]).round(4)
cmp = cmp.round(4).sort_values("pooled_rank_ic", ascending=False)
print("=== per-symbol view: pooled model vs per-symbol model ===")
print("positive d_ = per-symbol wins on that symbol\n")
print(cmp[["pooled_rank_ic", "per_sym_rank_ic", "d_rank_ic",
           "pooled_r2",      "per_sym_r2",      "d_r2",
           "pooled_dir_acc", "per_sym_dir_acc"]])

print("\n=== mean over symbols ===")
print(cmp[["pooled_rank_ic", "per_sym_rank_ic", "pooled_r2", "per_sym_r2",
           "pooled_dir_acc", "per_sym_dir_acc"]].mean().round(4))


=== per-symbol view: pooled model vs per-symbol model ===
positive d_ = per-symbol wins on that symbol

          pooled_rank_ic  per_sym_rank_ic  d_rank_ic  pooled_r2  per_sym_r2  \
symbol                                                                        
BTCUSDT           0.8412           0.7631    -0.0781     0.6321      0.3965   
ETHUSDT           0.8304           0.7002    -0.1302     0.6150      0.2235   
LTCUSDT           0.8274           0.7480    -0.0794     0.5838      0.1926   
AVAXUSDT          0.8228           0.7295    -0.0933     0.6085      0.3258   
XRPUSDT           0.7952           0.6988    -0.0964     0.6007      0.3759   
LINKUSDT          0.7725           0.7365    -0.0360     0.5368     -0.0889   
ADAUSDT           0.7698           0.7251    -0.0447     0.5562      0.3945   
SOLUSDT           0.7649           0.4711    -0.2938     0.3178     -1.5088   
DOGEUSDT          0.7546           0.6615    -0.0931     0.5173      0.2604   
BNBUSDT           0.6834   

## Frozen winner

### Model
- elasticnet, alpha=1e-4, l1_ratio=0.7, StandardScaler + symbol dummies.
- Pooled across all 10 symbols, symbol identity carried as one-hot dummies.
- Target: cumulative funding over the next 24 8h intervals (~8 days).

### Feature set (v5, 26 features)
Funding lags/EWMAs/vol, premium, clamp distance, cross-symbol spread, basket spread + z + rank, time features, regime indicators, kline aggregates, seven handpicked pair interactions. Full list in `FEATURE_COLS_V5`.

### Out-of-fold performance (5 purged/embargoed walk-forward folds, horizon=24)
| metric      | elasticnet_l1r0.7 | GBM lr0.02 l15 (ES) | persistence |
|-------------|-------------------|---------------------|-------------|
| R²          | 0.4545            | 0.4275              | -0.31       |
| rank IC     | 0.7199            | 0.7577              | 0.664       |
| dir_acc     | 0.883             | 0.916               | 0.843       |

R² lift over persistence is decisive; rank IC lift is modest (funding is already persistent). Signal is real.

**GBM (properly early-stopped) is a live alternative for the strategy input.** Larger rank IC and dir_acc, slightly smaller R². If the strategy gates on rank / direction rather than magnitude, GBM is preferable — pending an export path (`booster.dump_model()` codegens to nested if/else, or bind libLightGBM) and, more importantly, a re-run once the target is `net_funding_after_costs` from the honest sim.

### What the regime-gating experiments taught us
Gating the predictor's own output using hardcoded rules on `funding_sign_w90` or `funding_mean_w90` did **not** lift model quality on the retained trade subset in any way that justifies the mechanism at the model layer:

- **`gate_sign`** barely moved in-gate R² (0.50 → 0.51) while excluding 13% of rows, and the skipped-rows summary shows those rows had *negative* funding on average and the model correctly predicted *negative* on them. For a long-only carry strategy this is redundant with a strategy hurdle; for a bidirectional strategy it would kill exactly the short-carry opportunities you'd want.
- **`gate_mean_low_5e-5`** was neutral. Skipped rows had near-zero true funding and near-zero predictions — both agreed. Model quality unchanged; a strategy hurdle at that scale would do the same job.
- **`gate_mean_low_1e-4`** destroyed signal. Skipped 72% of rows including many with non-trivial positive funding; in-gate R² fell to 0.40.

**Read**: hardcoded regime filters on the predictor's own features aren't a real risk gate. The predictor already reflects regime through its inputs — a threshold on those inputs applied after the fact is either redundant (with a strategy hurdle) or destructive. A genuine risk gate answers a different question ("is this a bad regime to trade at all") with a different target and probably a different model class, and belongs in a separate workstream (see `funding_signal_pnl_plan.md`).

### Export
Freeze this here and hand off. Next step is exporting the coefficient set + scaler + symbol-dummy mapping as a JSON artifact for the C++ port; nothing further needed from this notebook.


## Export artifact

Freeze the v5 elasticnet winner as a self-contained JSON.

Steps:
1. Fit the winner per fold to check coefficient stability — a feature is "robust" if its coefficient is non-zero in at least 4/5 folds.
2. Drop non-robust features (symbol dummies always kept — they're identifiers, not signals).
3. Refit `Ridge(alpha=1.0)` on the pruned feature set over the full training window. Post-lasso OLS-style: L1 for selection, ridge for coefficient estimation.
4. Dump feature order, scaler mean/scale, coefficients, intercept, symbol-dummy order to `research/funding_signal_model.json`.

Prediction formula at inference: `y = ridge_coef @ ((x - scaler_mean) / scaler_scale) + intercept`, where `x` is `[selected_features..., symbol_dummies...]` in the artifact's order.


In [16]:
import json
from research.models import _with_symbol_dummies

# --- Step 1: fit per fold, collect coefficients ---
fold_coefs = []
fold_cols = None
for fold_i, (tr, _) in enumerate(CV.walk_forward_splits(feat_v5_clean, n_folds=5, horizon=HORIZON, embargo=5)):
    train = feat_v5_clean.iloc[tr]
    Xtr, cols = _with_symbol_dummies(train, FEATURE_COLS_V5)
    ytr = train[TARGET_COL].to_numpy()
    pipe = Pipeline([("sc", StandardScaler()),
                     ("m", ElasticNet(alpha=1e-4, l1_ratio=0.7, max_iter=20000))])
    pipe.fit(Xtr, ytr)
    fold_coefs.append(pipe["m"].coef_)
    fold_cols = cols

fold_coefs = np.array(fold_coefs)
nonzero_frac = (np.abs(fold_coefs) > 1e-10).mean(axis=0)

# --- Step 2: robustness-select features (symbol dummies always kept) ---
n_features = len(FEATURE_COLS_V5)
robust = nonzero_frac[:n_features] >= 0.8
selected_features = [f for f, r in zip(FEATURE_COLS_V5, robust) if r]

print("Feature robustness (fraction of folds with non-zero coef):")
for f, frac in sorted(zip(FEATURE_COLS_V5, nonzero_frac[:n_features]), key=lambda x: -x[1]):
    tag = "KEEP" if frac >= 0.8 else "DROP"
    print(f"  {f:35s} {frac:.2f}  {tag}")
print(f"\nKept {len(selected_features)}/{n_features} features (+ {len(fold_cols) - n_features} symbol dummies)")

# --- Step 3: refit ridge on selected features + full data ---
Xall, all_cols = _with_symbol_dummies(feat_v5_clean, selected_features)
yall = feat_v5_clean[TARGET_COL].to_numpy()

pipe_final = Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])
pipe_final.fit(Xall, yall)

scaler = pipe_final["sc"]
ridge = pipe_final["m"]

# --- Step 4: assemble JSON artifact ---
symbol_dummy_cols = [c for c in all_cols if c.startswith("sym_")]
symbol_dummy_symbols = [c.removeprefix("sym_") for c in symbol_dummy_cols]
baseline_symbol = sorted(set(feat_v5_clean["symbol"]) - set(symbol_dummy_symbols))[0]

artifact = {
    "target": TARGET_COL,
    "horizon_intervals": HORIZON,
    "feature_names": selected_features,
    "symbol_dummy_order": symbol_dummy_symbols,
    "symbol_dummy_baseline": baseline_symbol,
    "column_order": list(all_cols),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "ridge_coef": ridge.coef_.tolist(),
    "ridge_intercept": float(ridge.intercept_),
    "training_rows": int(len(feat_v5_clean)),
    "notes": ("Prediction = ridge_coef @ ((x - scaler_mean) / scaler_scale) + intercept. "
              "x = [selected_features..., symbol_dummies...] in column_order. Each symbol "
              "dummy is 1 if the row's symbol matches, else 0. The baseline symbol has all "
              "dummies zero. Prediction is cumulative funding over the next `horizon_intervals` 8h prints."),
}

out = REPO / "research" / "funding_signal_model.json"
out.write_text(json.dumps(artifact, indent=2))
print(f"\nartifact exported -> {out.relative_to(REPO)}  ({out.stat().st_size} bytes)")


Feature robustness (fraction of folds with non-zero coef):
  funding_lag1                        1.00  KEEP
  funding_ewma_h2                     1.00  KEEP
  funding_ewma_h6                     1.00  KEEP
  clamp_distance                      1.00  KEEP
  premium_trend_s3                    1.00  KEEP
  funding_mean_w90                    1.00  KEEP
  realized_return                     1.00  KEEP
  high_low_range                      1.00  KEEP
  taker_imb_x_lag1                    1.00  KEEP
  vol1m_x_clamp                       1.00  KEEP
  basket_z_funding_ewma_h6            1.00  KEEP
  funding_sign_w90                    0.80  KEEP
  funding_vol_rank_v30_r180           0.80  KEEP
  regime_x_lag1                       0.80  KEEP
  basket_x_lag1                       0.80  KEEP
  basket_rank_funding_lag1            0.80  KEEP
  basket_z_x_lag1                     0.80  KEEP
  realized_vol_1m                     0.60  DROP
  taker_imbalance                     0.40  DROP
  funding_